<a href="https://colab.research.google.com/github/Mays-Waddah/ASP-dynamic-list/blob/main/Python_verification_RSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import files
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature

# ---------------- Generate Keys - no base 64 ----------------
def generate_keys():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    public_key = private_key.public_key()

    with open("private_key.pem", "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open("public_key.pem", "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    # Download keys
    files.download("private_key.pem")
    files.download("public_key.pem")
    print("✔ Keys generated and downloaded: private_key.pem, public_key.pem")
    return private_key, public_key

# ---------------- Show File Hash ----------------
def get_file_hash(file_bytes):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(file_bytes)
    file_hash = digest.finalize()
    print("SHA-256 hash of file:", file_hash.hex())
    return file_hash

# ---------------- Sign File ----------------
def sign_file(private_key, file_bytes, signature_path="signature.sig"):
    signature = private_key.sign(
        file_bytes,
        padding.PKCS1v15(),
        hashes.SHA256()
    )
    with open(signature_path, "wb") as f:
        f.write(signature)

    # Download signature
    files.download(signature_path)
    print(f"✔ Signature signed and downloaded: {signature_path}")
    return signature

# ---------------- Verify File ----------------
def verify_file(public_key, file_bytes, signature_bytes):
    try:
        public_key.verify(signature_bytes, file_bytes, padding.PKCS1v15(), hashes.SHA256())
        print("✅ VALID SIGNATURE — Authentic and unchanged.")
    except InvalidSignature:
        print("❌ INVALID SIGNATURE — Not authentic.")

# ---------------- MAIN ----------------
print("Choose an option:\n1. Sign a file\n2. Verify a signature")
option = input("Enter 1 or 2: ").strip()

if option == "1":
    private_key, public_key = generate_keys()
    print("Upload a file to sign:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    # Show hash before signing
    get_file_hash(file_bytes)

    # Sign file
    sign_file(private_key, file_bytes)

elif option == "2":
    print("Upload the PUBLIC key (.pem):")
    uploaded_pub = files.upload()
    pub_key = serialization.load_pem_public_key(uploaded_pub[list(uploaded_pub.keys())[0]])

    print("Upload the ORIGINAL file to verify:")
    uploaded_file = files.upload()
    file_bytes = uploaded_file[list(uploaded_file.keys())[0]]

    print("Upload the SIGNATURE file (.sig):")
    uploaded_sig = files.upload()
    signature_bytes = uploaded_sig[list(uploaded_sig.keys())[0]]

    # Verify
    verify_file(pub_key, file_bytes, signature_bytes)

else:
    print("❌ Invalid option")


Choose an option:
1. Sign a file
2. Verify a signature
Enter 1 or 2: 2
Upload the PUBLIC key (.pem):


Saving public_key (4).pem to public_key (4).pem
Upload the ORIGINAL file to verify:


Saving message.txt to message (5).txt
Upload the SIGNATURE file (.sig):


Saving signature (1).sig to signature (1).sig
✅ VALID SIGNATURE — Authentic and unchanged.


In [13]:
from google.colab import files
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature
import base64
import time  # for measuring elapsed time

# ---------------- Generate Keys with Base 64 ----------------
def generate_keys():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    public_key = private_key.public_key()

    with open("private_key.pem", "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open("public_key.pem", "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    files.download("private_key.pem")
    files.download("public_key.pem")
    print("✔ Keys generated and downloaded: private_key.pem, public_key.pem")
    return private_key, public_key

# ---------------- Show File Hash ----------------
def get_file_hash(file_bytes):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(file_bytes)
    file_hash = digest.finalize()
    print("SHA-256 hash of file:", file_hash.hex())
    return file_hash

# ---------------- Sign File ----------------
def sign_file(private_key, file_bytes, signature_path="signature.b64"):
    # Measure only the signing operation
    start_time = time.time()
    signature = private_key.sign(file_bytes, padding.PKCS1v15(), hashes.SHA256())
    elapsed_algo = time.time() - start_time

    # Encode and save Base64 signature (excluded from measured time)
    signature_b64 = base64.b64encode(signature)
    with open(signature_path, "wb") as f:
        f.write(signature_b64)
    files.download(signature_path)

    print(f"✔ Signature signed, Base64-encoded, and downloaded: {signature_path}")
    print(f"Pure RSA signing time: {elapsed_algo:.6f} seconds")
    return signature_b64

# ---------------- Verify File ----------------
def verify_file(public_key, file_bytes, signature_b64):
    # Measure only the verification operation
    signature = base64.b64decode(signature_b64)
    start_time = time.time()
    try:
        public_key.verify(signature, file_bytes, padding.PKCS1v15(), hashes.SHA256())
        valid = True
        print("✅ VALID SIGNATURE — Authentic and unchanged.")
    except InvalidSignature:
        valid = False
        print("❌ INVALID SIGNATURE — Not authentic.")
    elapsed_algo = time.time() - start_time
    print(f"Pure RSA verification time: {elapsed_algo:.6f} seconds")
    return valid

# ---------------- MAIN ----------------
print("Choose an option:\n1. Sign a file\n2. Verify a signature")
option = input("Enter 1 or 2: ").strip()

if option == "1":
    private_key, public_key = generate_keys()
    print("Upload a file to sign:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    get_file_hash(file_bytes)
    sign_file(private_key, file_bytes)

elif option == "2":
    print("Upload the PUBLIC key (.pem):")
    uploaded_pub = files.upload()
    pub_key = serialization.load_pem_public_key(uploaded_pub[list(uploaded_pub.keys())[0]])

    print("Upload the ORIGINAL file to verify:")
    uploaded_file = files.upload()
    file_bytes = uploaded_file[list(uploaded_file.keys())[0]]

    print("Upload the SIGNATURE file (.b64):")
    uploaded_sig = files.upload()
    signature_b64 = uploaded_sig[list(uploaded_sig.keys())[0]]

    verify_file(pub_key, file_bytes, signature_b64)

else:
    print("❌ Invalid option")



Choose an option:
1. Sign a file
2. Verify a signature
Enter 1 or 2: 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Keys generated and downloaded: private_key.pem, public_key.pem
Upload a file to sign:


Saving message.txt to message (9).txt
SHA-256 hash of file: 80da19db55baf9e10da8f4eaeadcd7164819681b8a05573d9270a14af5a82ffa


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Signature signed, Base64-encoded, and downloaded: signature.b64
Pure RSA signing time: 0.001760 seconds


In [14]:
from google.colab import files
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature
import base64
import time  # for measuring elapsed time

# ---------------- Generate ECDSA Keys ----------------
def generate_keys():
    private_key = ec.generate_private_key(ec.SECP256R1())
    public_key = private_key.public_key()

    with open("private_key.pem", "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open("public_key.pem", "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    files.download("private_key.pem")
    files.download("public_key.pem")
    print("✔ ECDSA keys generated and downloaded: private_key.pem, public_key.pem")
    return private_key, public_key

# ---------------- Show File Hash ----------------
def get_file_hash(file_bytes):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(file_bytes)
    file_hash = digest.finalize()
    print("SHA-256 hash of file:", file_hash.hex())
    return file_hash

# ---------------- Sign File ----------------
def sign_file(private_key, file_bytes, signature_path="signature.b64"):
    # Measure only the ECDSA signing operation
    start_time = time.time()
    signature = private_key.sign(file_bytes, ec.ECDSA(hashes.SHA256()))
    elapsed_algo = time.time() - start_time

    # Encode and save Base64 signature (excluded from measured time)
    signature_b64 = base64.b64encode(signature)
    with open(signature_path, "wb") as f:
        f.write(signature_b64)
    files.download(signature_path)

    print(f"✔ Signature signed, Base64-encoded, and downloaded: {signature_path}")
    print(f"Pure ECDSA signing time: {elapsed_algo:.6f} seconds")
    return signature_b64

# ---------------- Verify File ----------------
def verify_file(public_key, file_bytes, signature_b64):
    signature = base64.b64decode(signature_b64)
    # Measure only the ECDSA verification operation
    start_time = time.time()
    try:
        public_key.verify(signature, file_bytes, ec.ECDSA(hashes.SHA256()))
        valid = True
        print("✅ VALID SIGNATURE — Authentic and unchanged.")
    except InvalidSignature:
        valid = False
        print("❌ INVALID SIGNATURE — Not authentic.")
    elapsed_algo = time.time() - start_time
    print(f"Pure ECDSA verification time: {elapsed_algo:.6f} seconds")
    return valid

# ---------------- MAIN ----------------
print("Choose an option:\n1. Sign a file\n2. Verify a signature")
option = input("Enter 1 or 2: ").strip()

if option == "1":
    private_key, public_key = generate_keys()
    print("Upload a file to sign:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    get_file_hash(file_bytes)
    sign_file(private_key, file_bytes)

elif option == "2":
    print("Upload the PUBLIC key (.pem):")
    uploaded_pub = files.upload()
    pub_key = serialization.load_pem_public_key(uploaded_pub[list(uploaded_pub.keys())[0]])

    print("Upload the ORIGINAL file to verify:")
    uploaded_file = files.upload()
    file_bytes = uploaded_file[list(uploaded_file.keys())[0]]

    print("Upload the SIGNATURE file (.b64):")
    uploaded_sig = files.upload()
    signature_b64 = uploaded_sig[list(uploaded_sig.keys())[0]]

    verify_file(pub_key, file_bytes, signature_b64)

else:
    print("❌ Invalid option")


Choose an option:
1. Sign a file
2. Verify a signature
Enter 1 or 2: 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ ECDSA keys generated and downloaded: private_key.pem, public_key.pem
Upload a file to sign:


Saving message.txt to message (10).txt
SHA-256 hash of file: 80da19db55baf9e10da8f4eaeadcd7164819681b8a05573d9270a14af5a82ffa


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔ Signature signed, Base64-encoded, and downloaded: signature.b64
Pure ECDSA signing time: 0.000190 seconds
